# Pick Value Curve Analysis

## Cross-Positional Opportunity Cost using Expected VOR

This notebook visualizes and validates the **pick-value curve** - the expected VOR at each draft position based on historical ADP data.

### Why This Matters:
- Enables **cross-positional opportunity cost** comparisons
- Answers: "Should I draft QB5 at pick 40, or wait for RB15?"
- Identifies true "steals" and "reaches" relative to market expectations

### Methodology:
1. Map preseason ADP to expected VOR projections
2. Smooth using LOESS-style rolling window (±10 picks)
3. Create position-agnostic value curve
4. Validate monotonicity and residuals

Let's visualize the curve and validate the model!

In [ ]:
import duckdb
import plotly.graph_objects as go
import plotly.express as px

conn = duckdb.connect('../data/warehouse.duckdb', read_only=True)
print("✅ Connected to warehouse")

## 1. Load Pick Value Curve Data

In [ ]:
# Load expected value by pick
pick_value = conn.execute("""
    SELECT 
        pick_no,
        round,
        expected_vor_at_pick,
        expected_vor_p25,
        expected_vor_p75,
        expected_vor_mean,
        expected_vor_stddev,
        expected_value_tier,
        position_recommendation,
        best_qb_vor,
        best_rb_vor,
        best_wr_vor,
        best_te_vor
    FROM int_expected_value_by_pick
    WHERE pick_no <= 140
    ORDER BY pick_no
""").df()

print(f"Loaded {len(pick_value)} picks")
print("\nValue tiers:")
print(pick_value['expected_value_tier'].value_counts())
pick_value.head(10)

## 2. Visualize the Pick Value Curve

The main curve showing expected VOR by draft pick with confidence bands.

In [ ]:
fig = go.Figure()

# Add confidence band (25th-75th percentile)
fig.add_trace(go.Scatter(
    x=pick_value['pick_no'],
    y=pick_value['expected_vor_p75'],
    mode='lines',
    line=dict(width=0),
    showlegend=False,
    hoverinfo='skip'
))

fig.add_trace(go.Scatter(
    x=pick_value['pick_no'],
    y=pick_value['expected_vor_p25'],
    mode='lines',
    line=dict(width=0),
    fillcolor='rgba(68, 138, 255, 0.2)',
    fill='tonexty',
    name='25th-75th percentile',
    hoverinfo='skip'
))

# Add main curve (median)
fig.add_trace(go.Scatter(
    x=pick_value['pick_no'],
    y=pick_value['expected_vor_at_pick'],
    mode='lines',
    line=dict(color='#448AFF', width=3),
    name='Expected VOR (Median)',
    hovertemplate='Pick %{x}<br>Expected VOR: %{y:.1f}<extra></extra>'
))

# Add round boundaries
for round_num in range(1, 15):
    pick_no = round_num * 10
    if pick_no <= 140:
        fig.add_vline(
            x=pick_no,
            line_dash='dash',
            line_color='gray',
            opacity=0.3,
            annotation_text=f"Rd {round_num+1}",
            annotation_position="top"
        )

fig.update_layout(
    title='Fantasy Draft Pick Value Curve (Expected VOR by Pick)',
    xaxis_title='Draft Pick Number',
    yaxis_title='Expected Value Over Replacement (VOR)',
    height=600,
    hovermode='x unified',
    template='plotly_white'
)

fig.show()

print("\nKey Observations:")
print("=" * 60)
print(f"Pick #1 Expected VOR: {pick_value.iloc[0]['expected_vor_at_pick']:.1f}")
print(f"Pick #50 Expected VOR: {pick_value.iloc[49]['expected_vor_at_pick']:.1f}")
print(f"Pick #100 Expected VOR: {pick_value.iloc[99]['expected_vor_at_pick']:.1f}")
print(f"\nTotal drop from #1 to #100: {pick_value.iloc[0]['expected_vor_at_pick'] - pick_value.iloc[99]['expected_vor_at_pick']:.1f} VOR points")

## 3. Cross-Position Value Comparison

How does expected VOR vary by position at each pick?

In [ ]:
# Create position-specific value curves
fig = go.Figure()

positions = [
    ('QB', 'best_qb_vor', '#FF6B6B'),
    ('RB', 'best_rb_vor', '#4ECDC4'),
    ('WR', 'best_wr_vor', '#45B7D1'),
    ('TE', 'best_te_vor', '#FFA07A')
]

for pos_name, col, color in positions:
    fig.add_trace(go.Scatter(
        x=pick_value['pick_no'],
        y=pick_value[col],
        mode='lines',
        name=pos_name,
        line=dict(color=color, width=2),
        hovertemplate=f'{pos_name} - Pick %{{x}}<br>Best Available VOR: %{{y:.1f}}<extra></extra>'
    ))

fig.update_layout(
    title='Best Available VOR by Position and Pick Number',
    xaxis_title='Draft Pick Number',
    yaxis_title='Best Available VOR at Position',
    height=600,
    hovermode='x unified',
    template='plotly_white',
    legend_title='Position'
)

fig.show()

print("\nPosition Scarcity Insights:")
print("=" * 60)
# Find where each position "runs out" of high value
for pos_name, col, _ in positions:
    data = pick_value[[' pick_no', col]].dropna()
    if len(data) > 0:
        # Find pick where VOR drops below 30 (arbitrary threshold)
        low_value = data[data[col] < 30]
        if len(low_value) > 0:
            cliff_pick = low_value.iloc[0]['pick_no']
            print(f"{pos_name}: Value cliff at pick #{cliff_pick:.0f}")

## 4. Validation: Monotonicity Test

The curve should generally decrease (later picks = lower expected value).

In [ ]:
# Calculate pick-to-pick changes
pick_value['vor_change'] = pick_value['expected_vor_at_pick'].diff()

# Plot changes
fig = px.bar(
    pick_value[pick_value['pick_no'] <= 100],
    x='pick_no',
    y='vor_change',
    title='Pick-to-Pick VOR Change (Should be mostly negative)',
    labels={'vor_change': 'VOR Change from Previous Pick', 'pick_no': 'Pick Number'},
    color='vor_change',
    color_continuous_scale=['red', 'gray', 'green'],
    color_continuous_midpoint=0
)

fig.add_hline(y=0, line_dash='dash', line_color='black')
fig.update_layout(height=400, showlegend=False)
fig.show()

# Statistics
increases = pick_value[pick_value['vor_change'] > 0]
decreases = pick_value[pick_value['vor_change'] < 0]

print("\nMonotonicity Check:")
print("=" * 60)
print(f"Decreasing picks: {len(decreases)} ({len(decreases)/len(pick_value)*100:.1f}%)")
print(f"Increasing picks: {len(increases)} ({len(increases)/len(pick_value)*100:.1f}%)")
print("\nLargest increases (potential positional scarcity effects):")
print(increases.nlargest(5, 'vor_change')[['pick_no', 'round', 'vor_change', 'position_recommendation']])

## 5. Application: Identifying Reaches and Steals

Compare actual draft picks to the expected value curve.

In [ ]:
# Load actual draft results
actual_draft = conn.execute("""
    SELECT 
        dp.pick_no,
        dp.player_name,
        dp.position,
        evp.expected_vor_at_pick,
        fdp.risk_adjusted_scarcity_vor as actual_vor
    FROM fct_draft_performance fdp
    JOIN stg_draft_picks dp ON fdp.player_id = dp.player_id
    LEFT JOIN int_expected_value_by_pick evp ON dp.pick_no = evp.pick_no
    WHERE fdp.games_played > 0
    ORDER BY dp.pick_no
""").df()

# Calculate draft surplus/deficit
actual_draft['vor_surplus'] = actual_draft['actual_vor'] - actual_draft['expected_vor_at_pick']

# Scatter plot
fig = px.scatter(
    actual_draft,
    x='expected_vor_at_pick',
    y='actual_vor',
    color='position',
    hover_data=['player_name', 'pick_no', 'vor_surplus'],
    title='Actual VOR vs Expected VOR (Picks Above Line = Outperformers)',
    labels={'expected_vor_at_pick': 'Expected VOR', 'actual_vor': 'Actual VOR'},
    color_discrete_map={'QB': '#FF6B6B', 'RB': '#4ECDC4', 'WR': '#45B7D1', 'TE': '#FFA07A'}
)

# Add diagonal line (perfect match)
max_vor = max(actual_draft['expected_vor_at_pick'].max(), actual_draft['actual_vor'].max())
fig.add_trace(go.Scatter(
    x=[0, max_vor],
    y=[0, max_vor],
    mode='lines',
    line=dict(color='gray', dash='dash'),
    name='Perfect Match',
    showlegend=True
))

fig.update_layout(height=600)
fig.show()

print("\nBiggest Steals (Actual >> Expected):")
print("=" * 60)
print(actual_draft.nlargest(10, 'vor_surplus')[['pick_no', 'player_name', 'position', 'expected_vor_at_pick', 'actual_vor', 'vor_surplus']])

print("\n\nBiggest Reaches (Actual << Expected):")
print("=" * 60)
print(actual_draft.nsmallest(10, 'vor_surplus')[['pick_no', 'player_name', 'position', 'expected_vor_at_pick', 'actual_vor', 'vor_surplus']])